In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

In [ ]:
# point matplotlib ticks inwards
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"
# add top and right ticks
plt.rcParams["xtick.top"] = True
plt.rcParams["ytick.right"] = True
# use tex
plt.rcParams["text.usetex"] = True
plt.rcParams["text.latex.preamble"] = r"""
        \usepackage{libertine}
        \usepackage[libertine]{newtxmath}
"""

In [ ]:
cpt_datablocks = [
    # 'S003DM', #electron  mass diff
    'S003MMR',
    'S004MMR',
    'S004DT', #muon mean life ratio
    'S067CP2',
    'Q007CPT',
    'S008DM',
    'S008DT',
    'S010DT',
    'S010D1',
    'S010D4',
    'S010DM', #
    'S011DRE',
    'S011DIM',
    'S011YRE',
    'S011XRM', #
    'S011DGM',
    'S011DMM', #
    'S013CPT',
    'S016CMR', #
    'S032CPT',
    'S042TVI',
    'S042TVJ',
    'S042TVK',
    'S042TVL',
    'S042DT3', #
    'S042CT7', #
    'S042CT8', #
    'S042CTX', # ??
    'S042CTY', # ??
    'S086CT1', #
    'S086CT0', #
    'S086A03', #
    'S086A04', #
    'S086A00', #
    'S086A01', #
    'S043MD', #W boson mass diff
    'S016MMD',
    'S018DM',
    'S018DT',
    'S019DT',
    'S019MMD',
    'S022DMM',
    'S022DT',
    'S024DMM',
    'S040DT', #
    'M009D1', #
    'M009W16', #
    # B043D not CPT
]
seqq_datablocks = [
    'S011XRP',
    'S013REX',
    'S013IMX'
]
# S004MEC should be true due to charge universality
all_datablocks = cpt_datablocks # + seqq_datablocks

In [ ]:
# import pdg
# api = pdg.connect()
# measurements = list(api.get(cpt_datablocks[0]).get_measurements())
# measurements[1].get_value().is_limit

In [ ]:
QUERY = f"""
SELECT pdgid.description, pdgmeasurement.pdgid, pdgmeasurement.id, pdgmeasurement_values.column_name, pdgdata.value_type, pdgmeasurement.technique, pdgreference.publication_year, pdgid.data_type, pdgdata.in_summary_table, pdgdata.value, pdgmeasurement_values.value, pdgmeasurement_values.error_positive, pdgmeasurement_values.error_negative, pdgmeasurement_values.stat_error_positive, pdgmeasurement_values.stat_error_negative
FROM pdgmeasurement_values
    JOIN pdgmeasurement ON pdgmeasurement.id = pdgmeasurement_values.pdgmeasurement_id
    JOIN pdgid ON pdgid.id = pdgmeasurement.pdgid_id
    JOIN pdgdata ON pdgdata.pdgid_id = pdgid.id
    JOIN pdgreference ON pdgreference.id = pdgmeasurement.pdgreference_id
--     JOIN pdgparticle ON pdgparticle.pdgid = pdgid.parent_pdgid
WHERE pdgdata.edition = '2025' AND pdgmeasurement.pdgid IN ({','.join('?'*len(all_datablocks))})
"""

con = sqlite3.connect("../data/pdgall-2025-v0.2.1.sqlite")
cur = con.cursor()
res = cur.execute(QUERY, all_datablocks)
data = res.fetchall()
columns = [col[0] for col in res.description]
print(len(data), "measurements")
print(columns)

df = pd.DataFrame(
    data,
    columns=[
        "pdgid.description",
        "pdgid",
        "pdgmeasurement_id",
        "column_name",
        "type",
        "technique",
        "year",
        "data_type",
        "insummary",
        "avg",
        "measurement",
        "error_positive",
        "error_negative",
        "stat_error_positive",
        "stat_error_negative",
    ],
)

In [ ]:
df

# manually correct some values
correction_dict = {
    13162: [7.8e-18, 8.4e-18, 8.4e-18],
    16300: [8.3e-3, 7.685e-3, 7.685e-3],
    30188: [2e-9, 4e-9, 4e-9],
    35702: [-6e-4, 12e-4, 12e-4],
    37056: [-2.5e-5, 8.7e-5, 8.7e-5],
    13161: [-3e-18, 4e-18, 4e-18]
}
# for some reason this node is duplicated
df[df['pdgid'] == 'S086CT1'] = df[df['pdgid'] == 'S086CT1'].drop_duplicates(subset=['pdgmeasurement_id'])
# correct the measurement, error_positive, and error_negative for the measurements with measurement_id in the correction_dict
for measurement_id, values in correction_dict.items():
    df.loc[df['pdgmeasurement_id'] == measurement_id, 'measurement'] = values[0]
    df.loc[df['pdgmeasurement_id'] == measurement_id, 'error_positive'] = values[1]
    df.loc[df['pdgmeasurement_id'] == measurement_id, 'error_negative'] = values[2]

In [ ]:
df[df['error_positive'].isna()]

In [ ]:

# show non-unique rows
df[df.duplicated(subset=['pdgmeasurement_id'])]

In [ ]:
df = df[~df['error_positive'].isna()]
# len(df)
print(len(df['pdgmeasurement_id'].unique()))
print(len(df))

In [ ]:
list(df.iloc[np.argmax(abs_norm_resids)])

In [ ]:
np.max(abs_norm_resids)

In [ ]:
plt.figure(figsize=(4,3))

truth = np.zeros(len(df))
truth[df['pdgid'].isin(['S016CMR', 'S004DT', 'S040DT', 'S042DT3'])] = 1
resid = np.array(df['measurement'] - truth)
error = np.where(resid > 0, df['error_positive'], df['error_negative'])
bins = np.linspace(0, 4, 20, endpoint=True)
bar = plt.hist(np.abs(resid/error), density=True, bins=bins, color='grey', label='Absolute normalized residuals')
# get tops of bars
bar_tops = bar[0]
bin_centers = bar[1][:-1] + np.diff(bar[1])/2
# plot numbers of measurements in each bin at the top of each bar
counts = np.bincount(np.digitize(np.abs(resid/error), bins=bins))[1:]
for i, count in enumerate(counts):
    if i == len(bar_tops):
        break
    plt.text(bin_centers[i], bar_tops[i], str(count), ha='center', va='bottom')

abs_norm_resids = np.abs(resid/error)
p_values = 1-norm.cdf(abs_norm_resids)
# plt.show()
# plot |N(0,1)|
x = np.linspace(0, 4, 100)
plt.plot(x, norm.pdf(x)*2, color='red', label='$|N(0,1)|$ density')
plt.xlabel('z-score')
plt.ylabel('Density')
plt.ylim(0, plt.ylim()[1]*1.05)
plt.legend(frameon=False)
plt.title('Residuals in CPT violation searches')
plt.savefig('../figs/CPT-zscores.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(4,3))
zs = np.abs(resid/error)
zspace = np.linspace(0, 2.5, 1000)
plt.plot(zspace, [np.mean(zs > z) for z in zspace], color='grey', label='CPT violation residuals')
import json
pdg2025_dist = json.loads(open('../results/measurement_dist/pdg2025-both.json').read())
plt.plot(pdg2025_dist['zspace'], pdg2025_dist['pair'], color='black', label='PDG 2025 pair differences')
plt.plot(zspace, (1-norm.cdf(zspace))*2, color='red', linestyle='--', label='$|N(0,1)|$')
plt.xlabel('$z$')
plt.ylabel('$P(|Z|>z)$')
plt.title('CPT violation measurement tail probability')
# plt.yscale('log')
plt.legend(frameon=False)
plt.xlim(0, 2.5)
plt.ylim(0, 1)
plt.savefig('../figs/cpt/cpt-dist.pdf', bbox_inches='tight')
plt.savefig('../figs/cpt/cpt-dist.png', bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
bar_tops

In [ ]:
counts

In [ ]:
bar_tops

In [ ]:
truth

In [ ]:
n = len(df)
# number within 1sd
print(str(np.sum(abs_norm_resids <= 1)) + '/' + str(n))
# proportion within 1sd
print(np.mean(abs_norm_resids <= 1))

In [ ]:
from scipy.stats import binom
1-binom.cdf(np.sum(abs_norm_resids <= 1)-1, n, 0.6827)

In [ ]:
# proportion within 2sd
np.mean(abs_norm_resids < 2)

In [ ]:
binom.cdf(np.sum(abs_norm_resids > 2.05), n, 2*norm.cdf(-2.05))

In [ ]:
# number within 1.5sd
np.sum(abs_norm_resids <= 1.5)


In [ ]:
1 - binom.cdf(np.sum(abs_norm_resids <= 1.5)-1, n, 1-2*norm.cdf(-1.5))

In [ ]:
abs_norm_resids

In [ ]:
print(np.mean(abs_norm_resids))
print(np.sqrt(2/np.pi))


In [ ]:
print(np.mean(abs_norm_resids**2))
print(1)

In [ ]:
plt.hist(p_values, density=True, bins=np.linspace(0,0.5,10), color='black')
plt.axhline(2, color='red')

In [ ]:
from scipy.stats import kstest, uniform
kstest(p_values, uniform(0,0.5).cdf)

In [ ]:
df[df['error_positive'].isna()]

In [ ]:
QUERY = f"""
SELECT value, description FROM pdgdoc WHERE table_name = 'PDGDATA'
"""
res = cur.execute(QUERY)
data = res.fetchall()
data

In [ ]:
# bad ones in the dataset

if row['comment'].str.contains("Repl. by"):
    continue
if row['tecn'] == 'RVUE' and alone == False:
    continue
if row['document'] in ['LAI 2005A', 'AMBROSINO 2006E']:
    continue